# HW — Topic 4 · 분포 고르기 (랩 확장)

**확률통계 · Topic 4** | 배점 20점 | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`HW_학번_T04.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T04.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

---

⭐ **미니 프로젝트 1 안내가 이번 주에 배포된다.** 이 과제는 그 예행연습이다.
여기서 연습한 **데이터 → 분포 선택 → 검증** 절차를 프로젝트에서 그대로 쓴다.

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| 1 | 새벽 0–5시 구간에 Poisson 적합 | 8 |
| 2 | Binomial 과 Poisson 을 분산으로 구별 | 7 |
| 3 | 가챠 설계 — 기댓값 · 표준편차 · 천장 | 5 |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −2점.

## Part 0. 준비

이 셀을 먼저 실행한다. **시드는 바꾸지 말 것.**

> 💡 데이터를 인터넷에서 받아온다. 실패하면 같은 성질의 **합성 데이터**로 대체되니
> 오류가 나도 그대로 진행하면 된다 (채점에 영향 없음).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)   # ⚠️ 이 줄은 바꾸지 마세요

URL = "https://inetguru.github.io/prob-stat-pnuace/data/t04_server_log.csv"


def load_log():
    """서버 로그를 불러온다. 인터넷이 막히면 같은 성질의 합성 데이터로 대체한다."""
    try:
        import pandas as pd
        df = pd.read_csv(URL)
        return df["hour"].to_numpy(), df["requests"].to_numpy()
    except Exception as e:
        print(f"  (원본을 못 받아 합성 데이터로 대체합니다: {type(e).__name__})")
        g = np.random.default_rng(20260302)
        minute = np.arange(1440)
        lam = 1.0 + 8.0 * np.exp(-((minute - 780) / 260.0) ** 2)   # 낮에 많고 새벽에 적다
        return minute // 60, g.poisson(lam)


hour, requests = load_log()
print(f"분 단위 기록 {len(requests):,}개 · 전체 평균 {requests.mean():.2f}")

## 문제 1 — 새벽 구간에 Poisson 적합 (8점)

랩에서는 **10–15시** 구간을 썼다. 이번에는 **새벽 0–5시** 구간으로 같은 작업을 한다.

> 💡 새벽은 $\lambda$ 가 작아 값의 종류가 적다 (0, 1, 2, 3 정도).
> 히스토그램 막대가 몇 개 안 되는 것이 **정상**이다.

### ✏️ TODO 1 — 0–5시 구간을 잘라내 요약한다 (a)

`hour` 는 0 부터 23 까지의 시각이다. 새벽 0–5시는 **hour 가 0 이상 5 미만**인 구간이다.

In [ ]:
# TODO 1: hour 가 0 이상 5 미만인 구간만 골라내세요
#         힌트 - requests[(hour >= 0) & (hour < 5)]
night = requests[:1]        # <- 이 줄을 고치세요

mean_n = night.mean()
var_n = night.var()
ratio_n = var_n / mean_n if mean_n > 0 else 0.0

print(f"  표본 수      : {night.size}")
print(f"  평균         : {mean_n:.4f}")
print(f"  분산         : {var_n:.4f}")
print(f"  분산/평균    : {ratio_n:.4f}")

### ✏️ TODO 2 — Poisson 을 적합해 겹쳐 그린다 (b)

$\hat{\lambda} = \bar{X}$ 로 두고 (STEP 3), 관측 히스토그램에 PMF 를 겹친다 (STEP 4).

> 📌 라벨은 **영어**로. Colab 에는 한글 폰트가 없어 글자가 □□□ 로 깨진다.

In [ ]:
lam_hat = mean_n
k = np.arange(0, max(night.max(), 1) + 1)
obs = np.bincount(night, minlength=k.size)[: k.size] / night.size

# TODO 2: lam_hat 을 쓰는 Poisson PMF 를 구하세요
#         힌트 - stats.poisson.pmf(k, lam_hat)
fit = np.zeros_like(k, dtype=float)      # <- 이 줄을 고치세요

plt.figure(figsize=(7.5, 4))
plt.bar(k, obs, color="#8b95a7", width=0.7, label="Observed (00:00-05:00)")
plt.plot(k, fit, "o-", color="#d97d17", lw=2, ms=5,
         label=f"Poisson(lambda = {lam_hat:.2f})")

plt.xlabel("requests per minute")
plt.ylabel("relative frequency")
plt.title("Night window: does a Poisson fit?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"  최대 차이 : {np.abs(obs - fit).max():.4f}")

### (c) 낮 시간대와 비교해 판단한다 — 2문장

분산/평균 비율을 근거로 쓸 것. 이 셀을 더블클릭해 작성한다.

> **답:** (여기에 작성)

## 문제 2 — 두 분포 구별하기 (7점)

평균이 거의 같은 두 데이터를 만든다. **어느 것이 Binomial 이고 어느 것이 Poisson 인지**
분산만 보고 판단한다.

### ✏️ TODO 3 — 평균과 분산을 표로 (a)(b)

In [ ]:
g = np.random.default_rng(20260302)
data1 = g.binomial(n=30, p=0.2, size=3000)
data2 = g.poisson(6, size=3000)

# TODO 3: 각 데이터의 평균 · 분산 · 분산/평균 비율을 출력하세요
#         힌트 - d.mean(), d.var(), d.var() / d.mean()
print(f"{'':<8}{'평균':>10}{'분산':>10}{'분산/평균':>12}")
print("-" * 42)
for name, d in (("data1", data1), ("data2", data2)):
    print(f"{name:<8}{0.0:>10.3f}{0.0:>10.3f}{0.0:>12.3f}")   # <- 이 줄을 고치세요

print()
print(f"  이론 - Binomial(30, 0.2) : 평균 {30*0.2:.1f}  분산 {30*0.2*0.8:.1f}")
print(f"  이론 - Poisson(6)        : 평균 6.0  분산 6.0")

### ✏️ TODO 4 — 두 히스토그램을 나란히 (c)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=True)

# TODO 4: 두 데이터의 히스토그램을 각각 그리세요
#         힌트 - ax.hist(d, bins=np.arange(-0.5, d.max() + 1.5), density=True)
for ax, (name, d) in zip(axes, (("data1", data1), ("data2", data2))):
    ax.set_title(name)
    ax.set_xlabel("value")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("relative frequency")
plt.tight_layout()
plt.show()

### (b) 분산만으로 구별할 수 있는 이유를 쓴다

$np(1-p)$ 와 $\lambda$ 를 비교해 설명할 것.

> **답:** (여기에 작성)

## 문제 3 — 가챠 설계 (5점)

어떤 게임의 ★5 확률이 $p = 0.006$ 으로 매번 일정하다.

### ✏️ TODO 5 — 기댓값 · 표준편차 · 300번 실패 확률 (a)(b)

첫 성공까지의 **시도 횟수**를 세는 정의를 쓴다.

$$\mathbb{E}[X] = \frac{1}{p} \qquad
\mathrm{sd}[X] = \frac{\sqrt{1-p}}{p} \qquad
\mathbb{P}[X > n] = (1-p)^{n}$$

In [ ]:
p = 0.006

# TODO 5: 아래 네 값을 구하세요
#         힌트 - 기댓값 1/p · 표준편차 (1-p)**0.5 / p · n번 모두 실패 (1-p)**n
ex = 0.0        # <- 기댓값
sd = 0.0        # <- 표준편차
fail300 = 0.0   # <- 300번 모두 실패할 확률
fail200 = 0.0   # <- 200번 모두 실패할 확률

print(f"  기댓값          : {ex:>8.2f} 회")
print(f"  표준편차        : {sd:>8.2f} 회")
print(f"  300번 모두 실패 : {fail300:>8.4f}   ({fail300*100:.1f}%)")
print(f"  200번 모두 실패 : {fail200:>8.4f}   ({fail200*100:.1f}%)")

### (c) 천장(pity) 시스템이 필요한 이유 — 3문장

위 (a)(b) 의 **숫자를 근거로** 쓴다. 이 셀을 더블클릭해 작성한다.

> 💡 힌트가 되는 물음 — 표준편차가 기댓값과 비슷하다는 것은 무슨 뜻인가?
> 평균만큼 뽑았는데도 못 얻는 사람이 얼마나 되는가?

> **답:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] `TODO 1~5` 를 모두 채웠다
- [ ] 문제 1(c) · 2(b) · 3(c) 의 **서술을 작성했다**
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 그래프에 축 이름 · 제목 · 범례가 있고 **한글이 없다**
- [ ] 파일명을 **`HW_학번_T04.ipynb`** 로 바꿨다

**제출처와 기한은 PLATO · Google Classroom 공지를 확인한다.**

### 자주 하는 실수

- Poisson 을 적합해 놓고 **분산/평균 비율을 안 재는 것** — 그게 검증 단계다
- Geometric 의 두 정의를 섞는 것 — 이 문제는 **시도 횟수**를 센다
- 그래프 축 라벨을 한글로 → Colab 에서 깨진다